# Прогноз остатков с LSTM и Optuna

На входе `df` с `REP_DT` и `OST_VAL`. Одна LSTM читает последнее окно наблюдений и выдаёт прогнозы на 1–12 календарных месяцев. Сначала обучаем базовую LSTM, затем подбираем параметры в Optuna и сравниваем обе версии на тесте.

Установи зависимости в отдельной ячейке при необходимости. `%pip install "torch>=2.10" optuna "mlflow>=3.18,<4" "pandas>=2.2" matplotlib`

## 1. Данные и настройки

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import optuna
import pandas as pd
import torch
from mlflow.models import ModelSignature
from mlflow.types import Schema, TensorSpec
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Загрузи свой df или используй файл проекта с CURRENCY = "RUB".
# df = pd.read_csv("df_fl_synthetic.csv").drop_duplicates(["REP_DT", "CURENCY"])

CURRENCY = None
HISTORY_COLS = []      # Числовые признаки, доступные на дату строки, например ["stat_roll_7d_median"].
LOOKBACK = None        # По умолчанию 60 дней или 6 месяцев.
HORIZONS = list(range(1, 13))
N_FOLDS = 3
TEST_START = None      # Дата первого выпуска test. По умолчанию выбирается из данных.
EPOCHS = 20
N_TRIALS = 10
YEAR_END_DAYS = 1      # Поставь 3, если пик бывает на последнем рабочем дне декабря.
YEAR_END_WEIGHT = 5.0  # Вес конца года при обучении. 1 означает обычную MAE.
SEED = 42
BASE_PARAMS = dict(hidden=32, layers=1, dropout=0.1, lr=0.001, batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)
print(f"Устройство {device}")

In [ ]:
data = df.copy()
currency_col = next((col for col in ("CURENCY", "CURRENCY") if col in data), None)
if currency_col:
    if CURRENCY is None and data[currency_col].nunique() > 1:
        raise ValueError("Укажи CURRENCY в настройках.")
    data = data[data[currency_col].notna()] if CURRENCY is None else data[data[currency_col] == CURRENCY]
    data = data.copy()

dates = pd.to_datetime(data["REP_DT"], format="mixed", errors="coerce")
data["REP_DT"] = dates.dt.tz_localize(None).dt.normalize()
data = data.dropna(subset=["REP_DT"]).set_index("REP_DT").sort_index()
if data.empty or data.index.has_duplicates:
    raise ValueError("Нужен один ряд остатков с одной строкой на дату.")

monthly = data.index.to_period("M").is_unique
if monthly:
    data.index = data.index + pd.offsets.MonthEnd(0)
data = data.asfreq("ME" if monthly else "D")
index = data.index
lookback = LOOKBACK if LOOKBACK is not None else (6 if monthly else 60)
HORIZONS = sorted(set(HORIZONS))
if not HORIZONS or any(not isinstance(h, (int, np.integer)) or not 1 <= h <= 12 for h in HORIZONS):
    raise ValueError("HORIZONS должен содержать целые числа от 1 до 12.")
HORIZONS = [int(h) for h in HORIZONS]
if lookback < 2 or N_FOLDS < 2 or EPOCHS < 1 or N_TRIALS < 1 or YEAR_END_DAYS < 1 or YEAR_END_WEIGHT <= 0:
    raise ValueError("Проверь настройки окна, фолдов, обучения и веса конца года.")

history_cols = list(dict.fromkeys(["OST_VAL", *HISTORY_COLS]))
numeric = data[history_cols].apply(pd.to_numeric, errors="coerce").astype(float)
numeric = numeric.replace([np.inf, -np.inf], np.nan)
y = numeric["OST_VAL"]
history = numeric.ffill().to_numpy()

## 2. История, целевые даты и календарь

Каждое входное окно заканчивается на дату выпуска `REP_DT`. Для дневного ряда факт ищется на точную дату через нужное число месяцев. Для месячного ряда факт сопоставляется с целевым месяцем. Пропуски факта исключаются из обучения и метрик отдельно для каждого горизонта.

Годовой пик задаётся календарным флагом на **целевую** дату. Неизвестные будущие значения статистических признаков в сеть не подаются.

In [ ]:
origin_pos = np.flatnonzero(y.notna().to_numpy() & (np.arange(len(index)) >= lookback - 1))
origin_dates = index[origin_pos]
window_pos = origin_pos[:, None] - np.arange(lookback - 1, -1, -1)

target_dates = np.column_stack([
    (origin_dates + (pd.offsets.MonthEnd(h) if monthly else pd.DateOffset(months=h))).to_numpy()
    for h in HORIZONS
])
flat_dates = pd.DatetimeIndex(target_dates.ravel())
targets = y.reindex(flat_dates).to_numpy().reshape(target_dates.shape)
known = np.isfinite(targets)

angle = 2 * np.pi * flat_dates.month.to_numpy() / 12
days_to_year_end = (flat_dates + pd.offsets.YearEnd(0) - flat_dates).days.to_numpy()
year_end = (days_to_year_end < YEAR_END_DAYS).reshape(target_dates.shape)
future_names = ["horizon", "month_sin", "month_cos", "is_year_end"]
future = torch.tensor(np.column_stack([
    np.tile(HORIZONS, len(origin_dates)) / 12, np.sin(angle), np.cos(angle), year_end.ravel(),
]).reshape(*target_dates.shape, len(future_names)), dtype=torch.float32)

## 3. Временные фолды

Для обучения оставляем только факты, известные до первого выпуска соответствующего фолда. Валидационные факты должны быть известны к началу теста. Граница теста по умолчанию оставляет около 20% выпусков с известным фактом на максимальном горизонте. Нормализация вычисляется заново по прошлым данным каждого фолда.

In [ ]:
longest = np.flatnonzero(known[:, -1])
if len(longest) < 3:
    raise ValueError("Мало истории для максимального горизонта и выбранного окна.")
test_date = pd.Timestamp(TEST_START or origin_dates[longest[int(0.8 * len(longest))]])
test_date = test_date.tz_localize(None).normalize()

# К первому фолду должны быть известны факты хотя бы по трём длинным прогнозам.
warmup_end = pd.Timestamp(target_dates[longest[2], -1])
valid_rows = np.flatnonzero(
    (origin_dates > warmup_end)
    & (target_dates[:, -1] < test_date.to_datetime64())
    & known.any(axis=1)
)
test_rows = np.flatnonzero((origin_dates >= test_date) & known.any(axis=1))
if len(valid_rows) < N_FOLDS or not len(test_rows):
    raise ValueError("Мало истории для временных фолдов и теста. Проверь окно, HORIZONS и TEST_START.")

folds = np.array_split(valid_rows, N_FOLDS)
print(f"Первый выпуск test с {test_date.date()}, размеры valid {[len(rows) for rows in folds]}")

## 4. LSTM и обучение

LSTM читает окно истории. На каждом горизонте сеть предсказывает поправку к последнему известному остатку с учётом календаря целевой даты. Функция потерь — MAE по доступным фактам, с дополнительным весом конца года.

In [ ]:
class BalanceLSTM(nn.Module):
    def __init__(self, hidden, layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(len(history_cols), hidden, num_layers=layers, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden + len(future_names), hidden),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1),
        )

    def forward(self, past, calendar):
        output, _ = self.lstm(past)
        state = output[:, -1:, :].expand(-1, calendar.size(1), -1)
        delta = self.head(torch.cat([state, calendar], dim=-1)).squeeze(-1)
        return past[:, -1, :1] + delta


def prepare(cutoff):
    history_train = pd.DataFrame(history[index < cutoff])
    center = history_train.mean().fillna(0).to_numpy()
    scale = history_train.std(ddof=0).replace(0, 1).fillna(1).to_numpy()
    normalized = np.nan_to_num((history - center) / scale, nan=0.0)
    labels = known & (target_dates < cutoff.to_datetime64())
    return torch.tensor(normalized[window_pos], dtype=torch.float32), labels, center, scale


def fit(params, prepared):
    torch.manual_seed(SEED)
    past, labels, center, scale = prepared
    rows = np.flatnonzero(labels.any(axis=1))
    values = np.where(labels[rows], (targets[rows] - center[0]) / scale[0], 0)
    weights = labels[rows] * np.where(year_end[rows], YEAR_END_WEIGHT, 1)
    dataset = TensorDataset(
        past[rows], future[rows], torch.tensor(values, dtype=torch.float32),
        torch.tensor(weights, dtype=torch.float32),
    )
    batches = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True)
    model = BalanceLSTM(params["hidden"], params["layers"], params["dropout"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"])

    model.train()
    for _ in range(EPOCHS):
        for batch in batches:
            x, calendar, actual, weight = (value.to(device) for value in batch)
            error = (model(x, calendar) - actual).abs()
            loss = (error * weight).sum() / weight.sum()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    return model


@torch.no_grad()
def predict(model, prepared, rows):
    past, _, center, scale = prepared
    model.eval()
    model_device = next(model.parameters()).device
    result = [
        model(past[batch].to(model_device), future[batch].to(model_device)).cpu().numpy()
        for batch in np.array_split(rows, max(1, (len(rows) + 255) // 256))
    ]
    return np.concatenate(result) * scale[0] + center[0]


def prediction_rows(rows, predicted, mask, split, fold=None):
    i, h = np.nonzero(mask[rows])
    samples = rows[i]
    return pd.DataFrame({
        "REP_DT": origin_dates[samples], "target_dt": target_dates[samples, h],
        "months": np.asarray(HORIZONS)[h], "target": targets[samples, h],
        "forecast": predicted[i, h], "baseline": y.iloc[origin_pos[samples]].to_numpy(),
        "year_end": year_end[samples, h], "split": split, "fold": fold,
    })

In [ ]:
cv_data = [(rows, prepare(origin_dates[rows[0]])) for rows in folds]
valid_labels = known & (target_dates < test_date.to_datetime64())

def cross_validate(params, keep=False, trial=None):
    scores, parts = [], []
    for fold, (rows, prepared) in enumerate(cv_data, start=1):
        model = fit(params, prepared)
        prediction = predict(model, prepared, rows)
        mask = valid_labels[rows]
        scores.append(float(np.abs(prediction[mask] - targets[rows][mask]).mean()))
        if keep:
            if fold == 1:
                train_labels = prepared[1]
                train_rows = np.flatnonzero(train_labels.any(axis=1))
                parts.append(prediction_rows(
                    train_rows, predict(model, prepared, train_rows), train_labels, "train", fold
                ))
            parts.append(prediction_rows(rows, prediction, valid_labels, "valid", fold))
        if trial is not None:
            trial.report(float(np.mean(scores)), step=fold - 1)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return float(np.mean(scores)), parts


baseline_cv_mae, baseline_parts = cross_validate(BASE_PARAMS, keep=True)
print(f"Базовая LSTM · средняя MAE по фолдам {baseline_cv_mae:.2f}")

## 5. Подбор параметров в Optuna

Optuna минимизирует среднюю MAE по временным фолдам в единицах остатка. Первый trial использует параметры базовой модели. Слишком слабые варианты могут завершаться после первых фолдов. `test` используется после завершения поиска.

In [ ]:
if not mlflow.config.is_tracking_uri_set():
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("lstm-optuna-balances")
client = mlflow.MlflowClient()

def objective(trial):
    params = {
        "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
        "layers": trial.suggest_int("layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.35),
        "lr": trial.suggest_float("lr", 0.0003, 0.005, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }
    return cross_validate(params, trial=trial)[0]


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=3),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
)
study.enqueue_trial(BASE_PARAMS)
study.optimize(objective, n_trials=N_TRIALS)
best_params = study.best_params
with mlflow.start_run(run_name="optuna"):
    mlflow.log_params({**best_params, "trials": N_TRIALS, "epochs": EPOCHS})
    mlflow.log_metrics({"baseline_cv_MAE": baseline_cv_mae, "best_cv_MAE": study.best_value})
    mlflow.log_text(study.trials_dataframe().to_csv(index=False), "optuna_trials.csv")

print(f"Лучшая MAE по фолдам {study.best_value:.2f}")
display(best_params)

## 6. Тест, MAE и MAPE

Для каждой версии обучаем итоговую модель на фактах, известных до `test_date`. `train` показывает прогнозы внутри первого обучающего фолда, `valid` объединяет прогнозы валидационных фолдов. На всех выборках сравниваем с простым прогнозом по последнему остатку. MAPE считается по ненулевым фактам.

In [ ]:
def score(rows, by):
    error = (rows["target"] - rows["forecast"]).abs()
    baseline_error = (rows["target"] - rows["baseline"]).abs()
    denominator = rows["target"].abs().replace(0, np.nan)
    return rows.assign(
        error=error, ape=100 * error / denominator,
        baseline_error=baseline_error, baseline_ape=100 * baseline_error / denominator,
    ).groupby(by, as_index=False).agg(
        MAE=("error", "mean"), MAPE_pct=("ape", "mean"),
        baseline_MAE=("baseline_error", "mean"), baseline_MAPE_pct=("baseline_ape", "mean"),
        n=("target", "size"), n_MAPE=("ape", "count"),
    )


tuned_cv_mae, tuned_parts = cross_validate(best_params, keep=True)
test_data = prepare(test_date)

example = (torch.zeros(2, lookback, len(history_cols)), torch.zeros(2, len(HORIZONS), len(future_names)))
signature = ModelSignature(
    inputs=Schema([
        TensorSpec(np.dtype("float32"), (-1, *tensor.shape[1:]), name)
        for name, tensor in zip(("past", "calendar"), example)
    ]),
    outputs=Schema([TensorSpec(np.dtype("float32"), (-1, len(HORIZONS)))]),
)
models, run_ids, model_uris, tables = {}, {}, {}, []
for name, params, cv_mae, parts in [
    ("LSTM", BASE_PARAMS, baseline_cv_mae, baseline_parts),
    ("LSTM + Optuna", best_params, tuned_cv_mae, tuned_parts),
]:
    model = fit(params, test_data)
    test = prediction_rows(test_rows, predict(model, test_data, test_rows), known, "test")
    rows = pd.concat([*parts, test], ignore_index=True).assign(model=name)
    if not np.isfinite(rows["forecast"]).all():
        raise RuntimeError(f"{name} вернула некорректный прогноз.")
    tables.append(rows)
    scores = score(rows, ["months", "split"])

    with mlflow.start_run(run_name=name) as run:
        run_ids[name] = run.info.run_id
        mlflow.log_params({
            **params, "lookback": lookback, "epochs": EPOCHS, "test_start": str(test_date.date()),
            "year_end_days": YEAR_END_DAYS, "year_end_weight": YEAR_END_WEIGHT,
        })
        mlflow.log_metric("cv_MAE", cv_mae)
        for row in scores.to_dict("records"):
            prefix = f"{row['split']}/{row['months']}m"
            mlflow.log_metrics({f"{prefix}/{key}": row[key] for key in ("MAE", "MAPE_pct") if pd.notna(row[key])})
        mlflow.log_text(scores.to_csv(index=False), "metrics.csv")
        cv_scores = score(rows[rows["split"] == "valid"], ["months", "fold"])
        mlflow.log_text(cv_scores.to_csv(index=False), "cv_metrics.csv")
        _, _, center, scale = test_data
        mlflow.log_dict({
            "history_cols": history_cols, "future_cols": future_names, "lookback": lookback,
            "horizons": HORIZONS, "center": center.tolist(), "scale": scale.tolist(),
            "monthly": bool(monthly), "year_end_days": YEAR_END_DAYS,
        }, "preprocessing.json")
        model = model.cpu().eval()
        info = mlflow.pytorch.log_model(
            model, name="lstm", input_example=example, signature=signature, serialization_format="pt2",
            pip_requirements=[f"torch=={torch.__version__.split('+')[0]}"],
        )
        model_uris[name] = info.model_uri
    models[name] = model

predictions = pd.concat(tables, ignore_index=True)
metrics = score(predictions, ["model", "months", "split"])
cv_metrics = score(predictions.query("split == 'valid'"), ["model", "months", "fold"])
year_end_metrics = score(predictions.query("split == 'test' and year_end"), ["model", "months"])
display(metrics.query("split == 'test'"))
display(cv_metrics)
if year_end_metrics.empty:
    print("В тесте нет наблюдений с флагом конца года.")
else:
    display(year_end_metrics)

## 7. Факт и прогноз

Для каждого горизонта можно посмотреть все три выборки. На каждой панели факт и прогноз взяты на одни и те же даты целевого остатка.

Локальный MLflow открывается командой `mlflow ui --backend-store-uri sqlite:///mlflow.db`. Модель загружается через `mlflow.pytorch.load_model` и принимает окно нормализованной истории и календарь целевых дат. Параметры лежат в `preprocessing.json`. Прогноз в единицах остатка равен `raw_forecast * scale[0] + center[0]`.

In [ ]:
PLOT_MONTHS = 3 if 3 in HORIZONS else HORIZONS[0]
for name in models:
    rows = predictions.query("model == @name and months == @PLOT_MONTHS")
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), layout="constrained")
    fig.suptitle(f"{name} · горизонт {PLOT_MONTHS} мес.")
    for split, ax in zip(("train", "valid", "test"), axes):
        part = rows[rows["split"] == split].sort_values(["target_dt", "REP_DT"])
        ax.set_title(f"{split} · {len(part)} пар", loc="left")
        if part.empty:
            ax.text(0.5, 0.5, "Нет пар факта и прогноза", ha="center", transform=ax.transAxes)
            ax.set(xticks=[], yticks=[])
            continue
        ax.plot(part["target_dt"], part["target"], "o-", color="#185277", markersize=3, label="Факт")
        ax.plot(part["target_dt"], part["forecast"], "s--", color="#d45b26", markersize=3, label="Прогноз")
        ax.set_ylabel("OST_VAL")
        ax.legend(frameon=False)
        ax.grid(alpha=0.2)
    fig.supxlabel("Дата целевого остатка")
    plt.show()
    client.log_figure(run_ids[name], fig, f"forecast_{PLOT_MONTHS}m.png")